# Yahoo Finance Web Scrapping Notebook
This notebook will be the introduction to working with the yfinance api for webscrapping. We will predominantly be using this for finding ETF price and volume data but we will need to adjust it for divdends. The best thing I think we can do is divide into sectors, but also pull full index (SPY, QQQ), we will then need to find proxies for different maturity bonds (long and short) and the equivalent of a money market (1-3 month treasuries). It may be a good idea to pull currency data from this as well, possibly the DXY index of USD strength.

## Libraries

In [12]:
import numpy as np
import pandas as pd
import altair as alt  

import yfinance as yf

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

ALright, let's start with just the basic pulls. I am pretty sure we should be able to pull the data using a batch pull but if not, let's pull them indvidually and add them to dataframes via a horizontal merge on date. We will only be focusing on closing prices and volume traded in the day. We can decide any data tansformations we want to use in the future.

In [13]:
# Let's create a list of sectors
sectors = ["SPY","QQQ","XLF", "XLK", "XLU", "XLV", "XLE", "XLI", "XLB", "XLP", "XLY", "XLRE", "XLC"]
prices = yf.download(sectors, period='max', auto_adjust=True)[['Close','Volume']]

[*********************100%***********************]  13 of 13 completed


I'm going to separate the data pull from the data analysis so I don't have to continue to query the API as rate limiting is a known issue.

In [16]:
prices.dropna(inplace=True)
prices.head()



Price            Close                                               \
Ticker             QQQ         SPY        XLB        XLC        XLE   
Date                                                                  
2018-06-19  167.658081  245.218002  25.089115  46.472141  26.737789   
2018-06-20  168.848892  245.636292  25.007605  47.048855  26.855844   
2018-06-21  167.381897  244.096420  24.741669  46.760494  26.358591   
2018-06-22  167.010345  244.541428  25.101986  46.965126  26.884468   
2018-06-25  163.247589  241.212540  24.711637  45.997749  26.344271   

Price                                                              ...  \
Ticker            XLF        XLI        XLK        XLP       XLRE  ...   
Date                                                               ...   
2018-06-19  23.704649  64.433456  33.221355  41.704491  24.398001  ...   
2018-06-20  23.643976  64.477470  33.291134  41.745335  24.661348  ...   
2018-06-21  23.574636  63.667961  33.035324  41.827003  24.808512  ...   
2018-06-22  23.461966  63.887943  32.928356  42.170063  25.025379  ...   
2018-06-25  23.210621  63.078430  32.244671  42.382420  24.963421  ...   

Price          Volume                                                  \
Ticker            XLC         XLE         XLF         XLI         XLK   
Date                                                                    
2018-06-19    16600.0  25215200.0  50214700.0  17955400.0  25913400.0   
2018-06-20   190000.0  23311800.0  42833600.0  11975900.0  22931200.0   
2018-06-21   428700.0  30019600.0  75190900.0  15497800.0  29634200.0   
2018-06-22   181500.0  56072600.0  61389400.0  13643900.0  31385800.0   
2018-06-25  2509600.0  36760600.0  80746100.0  23091500.0  36055400.0   

Price                                                                 
Ticker             XLP       XLRE         XLU        XLV         XLY  
Date                                                                  
2018-06-19  12639200.0  2535800.0  33294800.0  6431700.0  12339200.0  
2018-06-20   9562500.0  3725000.0  24635400.0  5900800.0   7886000.0  
2018-06-21  13272700.0  2827000.0  25407400.0  7423400.0  13700800.0  
2018-06-22   9938300.0  2292900.0  25953400.0  9829800.0  11135800.0  
2018-06-25  22124200.0  6956700.0  42932400.0  9925100.0  24133600.0  

[5 rows x 26 columns]

Alright, That data pull should work pretty easily let's take a look at much historical data we have. I thinmk the biggest concern is the fact that if we drop NA's we only have sector data from 2018. This does not give us a lot of exposure do different market regimes. I image this is do to the relatively new explosion of ETFs, and I imagine fixed income ETF's may contribute more to this problem. We may want to consider other options as surrogates.

In [17]:
min_date = min(prices.index)
print(f"The earliest date in our data is {min_date}.")

The earliest date in our data is 2018-06-19 00:00:00.
